# Result Analysis — Revised Output-Compatible Notebook

This notebook is revised to consume the outputs produced by `model_train_test_revised.py`.

It is designed for the new experiment requirements:

- **Stratified 10-fold cross-validation** evidence from `cv_metrics.csv`, `cv_summary.csv`, `cv_history.csv`, and `cv_predictions.csv`.
- **Held-out test evaluation** from `metrics.json`, `predictions.csv`, `confusion_matrix.npy`, and embeddings.
- **Statistical testing** from `statistical_tests.csv`, or regenerated inside this notebook from paired fold-level CV metrics.
- **No manuscript-number fallback by default**. Tables should reflect the results actually produced by your experiment.

Expected folder structure:

```text
results/
  baseline_alexnet/
    metrics.json
    cv_metrics.csv
    cv_history.csv
    cv_predictions.csv
    cv_summary.csv
    predictions.csv
    confusion_matrix.npy
    embeddings.npy
    embeddings_labels.npy
  proposed/
    ... same files ...
  all_runs_summary.csv
  statistical_tests.csv
```


## 0. Setup

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc as sk_auc,
    precision_recall_curve,
    average_precision_score,
)

RESULTS_DIR = Path("./results")
FIG_DIR = Path("./figures")
TABLE_DIR = Path("./tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Change these only if your run names differ from model_train_test_revised.py
BASELINES = ["alexnet", "efficientnet_b0", "densenet121", "resnet50", "swin_t"]
BASELINE_RUNS = [f"baseline_{b}" for b in BASELINES]
BASELINE_LABELS = {
    "baseline_alexnet": "AlexNet",
    "baseline_efficientnet_b0": "EfficientNet-B0",
    "baseline_densenet121": "DenseNet121",
    "baseline_resnet50": "ResNet50",
    "baseline_swin_t": "Swin-T",
}

ABLATION_RUNS = ["ensemble", "cxp", "wam", "lea", "proposed"]
ABLATION_LABELS = {
    "ensemble": "Ensemble",
    "cxp": "Ensemble+CXP",
    "wam": "Ensemble+CXP+WAM",
    "lea": "Ensemble+CXP+WAM+LEA",
    "proposed": "Proposed",
}

METRIC_COLS = ["accuracy", "precision", "sensitivity", "specificity", "f1", "mcc", "auc"]
DISPLAY_METRICS = ["accuracy", "f1", "mcc", "auc"]

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_metrics(run):
    return read_json(RESULTS_DIR / run / "metrics.json", default=None)


def load_cv_metrics(run):
    p = RESULTS_DIR / run / "cv_metrics.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()


def load_cv_summary(run):
    p = RESULTS_DIR / run / "cv_summary.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()


def load_history(run="proposed"):
    return read_json(RESULTS_DIR / run / "history.json", default=None)


def load_predictions(run="proposed"):
    p = RESULTS_DIR / run / "predictions.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()


def available_runs():
    if not RESULTS_DIR.exists():
        return []
    return sorted([p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and (p / "metrics.json").exists()])


def metric_value(d, key):
    if d is None:
        return np.nan
    return float(d.get(key, np.nan))


def mean_std_from_cv(run, metric):
    summary = load_cv_summary(run)
    if not summary.empty and {"metric", "mean", "std"}.issubset(summary.columns):
        row = summary[summary["metric"] == metric]
        if len(row):
            return float(row.iloc[0]["mean"]), float(row.iloc[0]["std"])
    cv = load_cv_metrics(run)
    if not cv.empty and metric in cv.columns:
        return float(cv[metric].mean()), float(cv[metric].std(ddof=1))
    return np.nan, np.nan


def fmt4(x):
    return "--" if pd.isna(x) else f"{x:.4f}"


def fmt_mean_std(mean, std):
    if pd.isna(mean):
        return "--"
    if pd.isna(std):
        return f"{mean:.4f}"
    return f"{mean:.4f} ± {std:.4f}"


def build_result_table(runs, labels, table_name):
    rows = []
    for run in runs:
        m = load_metrics(run)
        cv = load_cv_metrics(run)
        if m is None and cv.empty:
            print(f"Missing output for {run}; skipping.")
            continue
        row = {
            "Run": run,
            "Model": labels.get(run, run),
            "CV Type": (m or {}).get("cv_type", "StratifiedKFold" if not cv.empty else "--"),
            "CV Folds": int((m or {}).get("cv_folds", cv["fold"].nunique() if (not cv.empty and "fold" in cv.columns) else 0)),
            "Test Used for Early Stopping": bool((m or {}).get("test_set_used_for_early_stopping", False)),
        }
        for metric in METRIC_COLS:
            test_v = metric_value(m, metric)
            cv_mean, cv_std = mean_std_from_cv(run, metric)
            row[f"Test {metric}"] = test_v
            row[f"CV {metric} mean"] = cv_mean
            row[f"CV {metric} std"] = cv_std
            row[f"CV {metric} mean±std"] = fmt_mean_std(cv_mean, cv_std)
        row["Inference Time (s)"] = metric_value(m, "inference_time_s")
        rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_csv(TABLE_DIR / f"{table_name}_numeric.csv", index=False)
        display_cols = ["Model", "CV Type", "CV Folds", "Test Used for Early Stopping"]
        for metric in DISPLAY_METRICS:
            display_cols += [f"Test {metric}", f"CV {metric} mean±std"]
        display_cols += ["Inference Time (s)"]
        pretty = df[display_cols].copy()
        for col in pretty.columns:
            if col.startswith("Test ") or col == "Inference Time (s)":
                pretty[col] = pretty[col].map(lambda x: "--" if pd.isna(x) else (f"{x:.2f}" if col == "Inference Time (s)" else f"{x:.4f}"))
        pretty.to_csv(TABLE_DIR / f"{table_name}_display.csv", index=False)
        return df, pretty
    return df, df

print("Available runs:", available_runs())
print("Results dir:", RESULTS_DIR.resolve())
print("Figures dir:", FIG_DIR.resolve())
print("Tables dir:", TABLE_DIR.resolve())

## 1. Output sanity check

This section verifies that the training script produced the expected files and that the results support your manuscript claim of **stratified 10-fold cross-validation** with no test-set leakage during early stopping.

In [ ]:
expected_files = [
    "metrics.json",
    "cv_metrics.csv",
    "cv_history.csv",
    "cv_predictions.csv",
    "cv_summary.csv",
    "history.json",
    "predictions.csv",
    "confusion_matrix.npy",
    "model.pt",
]

check_rows = []
for run in BASELINE_RUNS + ABLATION_RUNS:
    rd = RESULTS_DIR / run
    m = load_metrics(run)
    cv = load_cv_metrics(run)
    row = {
        "Run": run,
        "Folder Exists": rd.exists(),
        "CV Type": (m or {}).get("cv_type", "--"),
        "CV Folds in metrics.json": (m or {}).get("cv_folds", np.nan),
        "Unique folds in cv_metrics.csv": cv["fold"].nunique() if (not cv.empty and "fold" in cv.columns) else 0,
        "Rows in cv_metrics.csv": len(cv),
        "Test used for early stopping": (m or {}).get("test_set_used_for_early_stopping", "--"),
    }
    for fname in expected_files:
        row[fname] = (rd / fname).exists()
    check_rows.append(row)

sanity_check = pd.DataFrame(check_rows)
sanity_check.to_csv(TABLE_DIR / "output_sanity_check.csv", index=False)
sanity_check

In [ ]:
# Compact warning summary
if sanity_check.empty:
    print("No result folders found. Run model_train_test_revised.py first.")
else:
    missing = sanity_check[(sanity_check["Folder Exists"] == False) | (sanity_check["Unique folds in cv_metrics.csv"] != 10)]
    leakage = sanity_check[sanity_check["Test used for early stopping"] == True]
    if len(missing):
        print("Runs that are missing or not 10-fold yet:")
        display(missing[["Run", "Folder Exists", "Unique folds in cv_metrics.csv", "Rows in cv_metrics.csv"]])
    else:
        print("All checked runs have 10 folds in cv_metrics.csv.")
    if len(leakage):
        print("WARNING: These runs report test-set use during early stopping:")
        display(leakage[["Run", "Test used for early stopping"]])
    else:
        print("No checked run reports test-set use during early stopping.")

## 2. Table 1 — Baseline comparison from live outputs

In [ ]:
table1_numeric, table1 = build_result_table(BASELINE_RUNS, BASELINE_LABELS, "table1_baselines")
table1

## 3. Table 2 — Ablation comparison from live outputs

In [ ]:
table2_numeric, table2 = build_result_table(ABLATION_RUNS, ABLATION_LABELS, "table2_ablations")
table2

In [ ]:
# Ablation trajectory based on held-out test metrics when available, otherwise CV means.
if not table2_numeric.empty:
    plot_df = table2_numeric.copy()
    x = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for metric in ["accuracy", "f1", "mcc"]:
        y = plot_df[f"Test {metric}"].to_numpy(dtype=float)
        # If a test metric is missing, use the CV mean.
        y = np.where(np.isfinite(y), y, plot_df[f"CV {metric} mean"].to_numpy(dtype=float))
        ax.plot(x, y, marker="o", label=metric.upper() if metric != "f1" else "F1")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Model"], rotation=25, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Ablation trajectory")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ablation_trajectory.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No ablation outputs found.")

## 4. Statistical testing

The revised training script writes `results/statistical_tests.csv` after `--run_all`. If that file is missing, this notebook regenerates it from the paired fold-level `cv_metrics.csv` files.

Recommended manuscript interpretation: use the **Wilcoxon p-value** as the main non-parametric test and report paired t-test/Cohen's dz as supplementary evidence.

In [ ]:
def compute_paired_statistical_tests(reference="proposed", metrics=("accuracy", "f1", "mcc", "auc")):
    try:
        from scipy.stats import wilcoxon, ttest_rel
        has_scipy = True
    except Exception as e:
        print("SciPy not available; cannot compute Wilcoxon or paired t-test:", e)
        has_scipy = False

    ref = load_cv_metrics(reference)
    if ref.empty:
        print(f"Missing reference CV metrics: {RESULTS_DIR / reference / 'cv_metrics.csv'}")
        return pd.DataFrame()

    rows = []
    for run in available_runs():
        if run == reference:
            continue
        comp = load_cv_metrics(run)
        if comp.empty:
            continue
        merged = ref.merge(comp, on="fold", suffixes=("_reference", "_comparison"))
        if merged.empty:
            continue
        for metric in metrics:
            ref_col = f"{metric}_reference"
            cmp_col = f"{metric}_comparison"
            if ref_col not in merged.columns or cmp_col not in merged.columns:
                continue
            x = merged[ref_col].astype(float).to_numpy()
            y = merged[cmp_col].astype(float).to_numpy()
            mask = np.isfinite(x) & np.isfinite(y)
            x, y = x[mask], y[mask]
            if len(x) < 2:
                continue
            diff = x - y
            wilcoxon_stat = wilcoxon_p = paired_t_stat = paired_t_p = np.nan
            if has_scipy:
                try:
                    if np.allclose(diff, 0):
                        wilcoxon_stat, wilcoxon_p = 0.0, 1.0
                    else:
                        res = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
                        wilcoxon_stat, wilcoxon_p = float(res.statistic), float(res.pvalue)
                except Exception as e:
                    print(f"Wilcoxon failed for {reference} vs {run}, {metric}: {e}")
                try:
                    res_t = ttest_rel(x, y, nan_policy="omit")
                    paired_t_stat, paired_t_p = float(res_t.statistic), float(res_t.pvalue)
                except Exception as e:
                    print(f"Paired t-test failed for {reference} vs {run}, {metric}: {e}")
            sd = np.std(diff, ddof=1)
            cohen_dz = np.mean(diff) / sd if np.isfinite(sd) and not np.isclose(sd, 0) else np.nan
            rows.append({
                "reference": reference,
                "comparison": run,
                "metric": metric,
                "n_pairs": int(len(x)),
                "reference_mean": float(np.mean(x)),
                "comparison_mean": float(np.mean(y)),
                "mean_difference_reference_minus_comparison": float(np.mean(diff)),
                "wilcoxon_statistic": wilcoxon_stat,
                "wilcoxon_p_value": wilcoxon_p,
                "paired_t_statistic": paired_t_stat,
                "paired_t_p_value": paired_t_p,
                "cohen_dz": float(cohen_dz) if np.isfinite(cohen_dz) else np.nan,
                "significant_at_0_05": bool(np.isfinite(wilcoxon_p) and wilcoxon_p < 0.05),
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(RESULTS_DIR / "statistical_tests.csv", index=False)
    return out

stat_path = RESULTS_DIR / "statistical_tests.csv"
if stat_path.exists():
    statistical_tests = pd.read_csv(stat_path)
else:
    statistical_tests = compute_paired_statistical_tests(reference="proposed")

if not statistical_tests.empty:
    statistical_tests = statistical_tests.sort_values(["metric", "comparison"]).reset_index(drop=True)
    statistical_tests.to_csv(TABLE_DIR / "statistical_tests.csv", index=False)
    display(statistical_tests)
else:
    print("No statistical tests available yet. Run all models with --run_all, or ensure each run has cv_metrics.csv.")

In [ ]:
# Manuscript-friendly compact statistical table
if "statistical_tests" in globals() and not statistical_tests.empty:
    compact_stats = statistical_tests.copy()
    compact_stats["Wilcoxon p"] = compact_stats["wilcoxon_p_value"].map(lambda x: "--" if pd.isna(x) else f"{x:.4g}")
    compact_stats["Paired t-test p"] = compact_stats["paired_t_p_value"].map(lambda x: "--" if pd.isna(x) else f"{x:.4g}")
    compact_stats["Δ mean"] = compact_stats["mean_difference_reference_minus_comparison"].map(lambda x: "--" if pd.isna(x) else f"{x:.4f}")
    compact_stats["Cohen dz"] = compact_stats["cohen_dz"].map(lambda x: "--" if pd.isna(x) else f"{x:.3f}")
    compact_stats["Significant"] = compact_stats["significant_at_0_05"].map(lambda x: "Yes" if bool(x) else "No")
    compact_stats = compact_stats[["comparison", "metric", "n_pairs", "Δ mean", "Wilcoxon p", "Paired t-test p", "Cohen dz", "Significant"]]
    compact_stats.to_csv(TABLE_DIR / "statistical_tests_compact.csv", index=False)
    compact_stats
else:
    print("No compact statistical table generated.")

## 5. Cross-validation summary and fold-level plots

In [ ]:
# Proposed model CV summary
prop_cv_summary = load_cv_summary("proposed")
if prop_cv_summary.empty:
    prop_cv = load_cv_metrics("proposed")
    if not prop_cv.empty:
        prop_cv_summary = pd.DataFrame([
            {"variant": "proposed", "metric": m, "mean": prop_cv[m].mean(), "std": prop_cv[m].std(ddof=1), "folds": prop_cv["fold"].nunique()}
            for m in METRIC_COLS if m in prop_cv.columns
        ])

if not prop_cv_summary.empty:
    prop_cv_summary.to_csv(TABLE_DIR / "proposed_cv_summary.csv", index=False)
    display(prop_cv_summary)
else:
    print("No proposed cv_summary.csv or cv_metrics.csv found.")

In [ ]:
# Fold-level F1 comparison across available runs
fold_rows = []
for run in BASELINE_RUNS + ABLATION_RUNS:
    cv = load_cv_metrics(run)
    if not cv.empty and "f1" in cv.columns:
        temp = cv[["fold", "f1"]].copy()
        temp["Run"] = run
        temp["Model"] = BASELINE_LABELS.get(run, ABLATION_LABELS.get(run, run))
        fold_rows.append(temp)

if fold_rows:
    fold_f1 = pd.concat(fold_rows, ignore_index=True)
    plt.figure(figsize=(10, 4.5))
    labels = [BASELINE_LABELS.get(r, ABLATION_LABELS.get(r, r)) for r in BASELINE_RUNS + ABLATION_RUNS if r in fold_f1["Run"].unique()]
    data = [fold_f1.loc[fold_f1["Model"] == label, "f1"].to_numpy() for label in labels]
    plt.boxplot(data, labels=labels, showmeans=True)
    plt.ylabel("Fold-level F1-score")
    plt.title("10-fold CV F1-score distribution")
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "cv_fold_f1_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No fold-level F1 values found.")

## 6. Training history

In [ ]:
# Final-fit training history for the proposed model
hist = load_history("proposed")
if hist:
    epochs = np.arange(1, len(hist.get("train_acc", [])) + 1)
    if len(epochs):
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(epochs, hist["train_acc"], marker="o", label="Train")
        ax.plot(epochs, hist["val_acc"], marker="s", label="Validation")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy")
        ax.set_title("Final-fit training accuracy")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "final_training_accuracy.png", dpi=300, bbox_inches="tight")
        plt.show()

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(epochs, hist["train_loss"], marker="o", label="Train")
        ax.plot(epochs, hist["val_loss"], marker="s", label="Validation")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Final-fit training loss")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "final_training_loss.png", dpi=300, bbox_inches="tight")
        plt.show()

        print("Validation split used:", hist.get("val_split_used", "unknown"))
        print("Test set used for early stopping:", hist.get("test_set_used_for_early_stopping", "unknown"))
    else:
        print("history.json exists, but contains no epoch values.")
else:
    print("No proposed/history.json found.")

In [ ]:
# Cross-validation training history, averaged across folds
cvh_path = RESULTS_DIR / "proposed" / "cv_history.csv"
if cvh_path.exists():
    cvh = pd.read_csv(cvh_path)
    required = {"epoch", "train_acc", "val_acc", "train_loss", "val_loss"}
    if required.issubset(cvh.columns):
        agg = cvh.groupby("epoch").agg({
            "train_acc": ["mean", "std"],
            "val_acc": ["mean", "std"],
            "train_loss": ["mean", "std"],
            "val_loss": ["mean", "std"],
        })
        agg.columns = ["_".join(c).strip() for c in agg.columns]
        agg = agg.reset_index()
        agg.to_csv(TABLE_DIR / "proposed_cv_history_mean_std.csv", index=False)

        fig, ax = plt.subplots(figsize=(6, 4))
        for prefix, label in [("train_acc", "Train"), ("val_acc", "Validation")]:
            mean = agg[f"{prefix}_mean"].to_numpy(dtype=float)
            std = agg[f"{prefix}_std"].fillna(0).to_numpy(dtype=float)
            ep = agg["epoch"].to_numpy(dtype=float)
            ax.plot(ep, mean, marker="o", label=label)
            ax.fill_between(ep, mean - std, mean + std, alpha=0.15)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy")
        ax.set_title("10-fold CV training accuracy, mean ± std")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "cv_training_accuracy_mean_std.png", dpi=300, bbox_inches="tight")
        plt.show()

        fig, ax = plt.subplots(figsize=(6, 4))
        for prefix, label in [("train_loss", "Train"), ("val_loss", "Validation")]:
            mean = agg[f"{prefix}_mean"].to_numpy(dtype=float)
            std = agg[f"{prefix}_std"].fillna(0).to_numpy(dtype=float)
            ep = agg["epoch"].to_numpy(dtype=float)
            ax.plot(ep, mean, marker="o", label=label)
            ax.fill_between(ep, mean - std, mean + std, alpha=0.15)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("10-fold CV training loss, mean ± std")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "cv_training_loss_mean_std.png", dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("cv_history.csv is missing required columns:", required - set(cvh.columns))
else:
    print("No proposed/cv_history.csv found.")

## 7. Held-out test results: confusion matrix, ROC, and precision–recall curves

In [ ]:
pred = load_predictions("proposed")
cm_path = RESULTS_DIR / "proposed" / "confusion_matrix.npy"

if cm_path.exists():
    cm = np.load(cm_path)
elif not pred.empty and {"y_true", "y_pred"}.issubset(pred.columns):
    cm = confusion_matrix(pred["y_true"], pred["y_pred"], labels=[0, 1])
else:
    cm = None

if cm is not None:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Pneumonia"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title("Held-out test confusion matrix")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No confusion matrix or predictions found for proposed run.")

In [ ]:
if not pred.empty and {"y_true", "prob_pneumonia"}.issubset(pred.columns):
    y_true = pred["y_true"].astype(int).to_numpy()
    y_score = pred["prob_pneumonia"].astype(float).to_numpy()

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = sk_auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Held-out test ROC curve")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, label=f"AP = {ap:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Held-out test precision–recall curve")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "precision_recall_curve.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No proposed predictions.csv with y_true and prob_pneumonia found.")

## 8. t-SNE of learned embeddings

In [ ]:
from sklearn.manifold import TSNE

emb_path = RESULTS_DIR / "proposed" / "embeddings.npy"
lab_path = RESULTS_DIR / "proposed" / "embeddings_labels.npy"

if emb_path.exists() and lab_path.exists():
    emb = np.load(emb_path)
    lab = np.load(lab_path)
    if len(emb) >= 5:
        perplexity = min(30, max(2, (len(emb) - 1) // 3))
        proj = TSNE(n_components=2, init="pca", perplexity=perplexity, random_state=42).fit_transform(emb)
        fig, ax = plt.subplots(figsize=(5.5, 5))
        for cls, name in [(0, "Normal"), (1, "Pneumonia")]:
            mask = lab == cls
            ax.scatter(proj[mask, 0], proj[mask, 1], s=10, alpha=0.7, label=name)
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
        ax.set_title("t-SNE of proposed model embeddings")
        ax.legend()
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "tsne_embeddings.png", dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("Not enough embeddings for t-SNE.")
else:
    print("embeddings.npy or embeddings_labels.npy not found for proposed run.")

## 9. Explainability panel — Grad-CAM, Score-CAM, LIME, and SHAP

This section imports the revised training module and loads `results/proposed/model.pt`. It is defensive: missing packages are skipped and replaced by placeholder panels.

Install optional packages when needed:

```bash
pip install grad-cam lime shap scikit-image
```


In [ ]:
import importlib.util
import sys

import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

TRAINING_SCRIPT_CANDIDATES = [
    Path("./model_train_test_revised.py"),
    Path("./model_train_test.py"),
]


def import_training_module():
    for script in TRAINING_SCRIPT_CANDIDATES:
        if script.exists():
            spec = importlib.util.spec_from_file_location("mtt", script)
            mtt = importlib.util.module_from_spec(spec)
            sys.modules["mtt"] = mtt
            spec.loader.exec_module(mtt)
            print(f"Imported training module from {script}")
            return mtt
    raise FileNotFoundError("Could not find model_train_test_revised.py or model_train_test.py in the notebook folder.")

mtt = import_training_module()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mtt.IMAGENET_MEAN, mtt.IMAGENET_STD),
])


def load_proposed_model():
    model = mtt.build_model("proposed").to(device).eval()
    ckpt = RESULTS_DIR / "proposed" / "model.pt"
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state)
    else:
        print("WARNING: proposed/model.pt not found. Using randomly initialized weights.")
    return model


def choose_sample_image():
    pred = load_predictions("proposed")
    if pred.empty:
        return None
    # Prefer a correctly classified pneumonia example. Fall back to any available image.
    if {"y_true", "y_pred", "path"}.issubset(pred.columns):
        hit = pred[(pred["y_true"] == 1) & (pred["y_pred"] == 1)]
        if len(hit):
            return hit.iloc[0]["path"]
        hit = pred[pred["y_true"] == 1]
        if len(hit):
            return hit.iloc[0]["path"]
        return pred.iloc[0]["path"]
    return None

SAMPLE_IMG = choose_sample_image()
print("Sample image:", SAMPLE_IMG)

In [ ]:
def to_display(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    return np.array(img)

panels, titles = [], []

if SAMPLE_IMG:
    rgb = to_display(SAMPLE_IMG)
    x = tf(Image.open(SAMPLE_IMG).convert("RGB")).unsqueeze(0).to(device)
    panels.append(rgb)
    titles.append("Input")
    model = load_proposed_model()

    # Grad-CAM and Score-CAM
    try:
        from pytorch_grad_cam import GradCAM, ScoreCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
        target_layer = model.resnet[-1][-1]
        rgb_norm = rgb.astype(np.float32) / 255.0
        for cam_cls, name in [(GradCAM, "Grad-CAM"), (ScoreCAM, "Score-CAM")]:
            try:
                cam = cam_cls(model=model, target_layers=[target_layer])
                grayscale = cam(input_tensor=x)[0]
                panels.append(show_cam_on_image(rgb_norm, grayscale, use_rgb=True))
                titles.append(name)
            except Exception as e:
                print(f"{name} failed: {e}")
                panels.append(np.full_like(rgb, 200))
                titles.append(name)
    except Exception as e:
        print("pytorch-grad-cam unavailable; CAM panels use placeholders:", e)
        for name in ["Grad-CAM", "Score-CAM"]:
            panels.append(np.full_like(rgb, 200))
            titles.append(name)

    # LIME
    try:
        from lime import lime_image
        from skimage.segmentation import mark_boundaries

        def batch_predict(images):
            model.eval()
            batch = torch.stack([tf(Image.fromarray(im.astype(np.uint8))) for im in images]).to(device)
            with torch.no_grad():
                return F.softmax(model(batch), dim=1).cpu().numpy()

        explainer = lime_image.LimeImageExplainer()
        exp = explainer.explain_instance(
            rgb,
            batch_predict,
            top_labels=2,
            hide_color=0,
            num_samples=200,
        )
        temp, mask = exp.get_image_and_mask(
            exp.top_labels[0],
            positive_only=True,
            num_features=5,
            hide_rest=False,
        )
        panels.append((mark_boundaries(temp / 255.0, mask) * 255).astype(np.uint8))
        titles.append("LIME")
    except Exception as e:
        print(f"LIME skipped: {e}")
        panels.append(np.full_like(rgb, 200))
        titles.append("LIME")

    # SHAP
    try:
        import shap
        background = x.repeat(4, 1, 1, 1)
        explainer = shap.GradientExplainer(model, background)
        shap_values = explainer.shap_values(x)
        sv = shap_values[1] if isinstance(shap_values, list) else shap_values
        sal = np.abs(sv[0]).sum(0)
        sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
        panels.append((plt.cm.coolwarm(sal)[..., :3] * 255).astype(np.uint8))
        titles.append("SHAP")
    except Exception as e:
        print(f"SHAP skipped: {e}")
        panels.append(np.full_like(rgb, 200))
        titles.append("SHAP")
else:
    print("No sample image available. Run the proposed model first, or check predictions.csv paths.")

In [ ]:
if panels:
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(2.8 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img)
        ax.set_title(title, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "explainability_panel.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", FIG_DIR / "explainability_panel.png")
else:
    print("No explainability panels generated.")

## 10. Export manuscript tables to LaTeX

In [ ]:
tables_to_export = {}
for name in ["table1", "table2", "compact_stats", "prop_cv_summary"]:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            tables_to_export[name] = obj

for name, tbl in tables_to_export.items():
    tex_path = TABLE_DIR / f"{name}.tex"
    tex = tbl.to_latex(index=False, na_rep="--", escape=False, float_format="%.4f")
    tex_path.write_text(tex, encoding="utf-8")
    print(f"Saved {tex_path}")
    print(f"% ===== {name} =====")
    print(tex)
